# Assignment #8


In [ ]:
#First, we install and import the libraries

#!pip install chardet
#!pip install geopandas

import chardet
import pandas as pd
from pandas import Series, DataFrame
import numpy as np
import matplotlib.pyplot as plt 
import geopandas as gpd
from geopandas import GeoSeries
from shapely.geometry import Point, LineString

In [ ]:
#!pip install folium
import geopandas as gpd
from geopandas import GeoSeries
from shapely.geometry import Point, LineString
import folium 
from folium import Marker, GeoJson
from folium.plugins import MarkerCluster, HeatMap
import matplotlib.pyplot as plt
import geopandas as gpd

1. Import data from the online source

In [ ]:
# Gettting the character format (encoding type)

base = open(r'../../_data/data_dengue_peru.csv', 'rb').read()
det = chardet.detect(base)
charenc = det['encoding']
charenc

In [ ]:
# Step 1: We get the encoding and format type of the characters that compose the dataset
directory = r'../../_data/data_dengue_peru.csv' #Definition of the directory
with open(directory, 'rb') as file: #Opening the file in binary mode
    content_file = file.read()

characters = chardet.detect(content_file) #Use Chardet to detect the coding
encoding = characters['encoding']

# Data importation
df_dengue = pd.read_csv(directory, encoding=encoding, dtype={'Ubigeo': 'str'}, low_memory=False) #read csv, parameters specified


df_dengue

2. Generate ubigeo for Departments and Provinces taking the first two and four numbers. 

In [ ]:
df_dengue['UBIPROV'] = df_dengue['Ubigeo'].astype(str).str[0:4].copy()
df_dengue['UBIDPTO'] = df_dengue['Ubigeo'].astype(str).str[0:2].copy()

df_dengue

3. Use geopandas to plot the number of cases in 2021 by the district using a continuous legend. Do not forget to indicate the color of NA values. Use the provided shapefile.


In [ ]:
# Download shape file at district level

maps = gpd.read_file(r'../../_data/LIMITE_DISTRITAL_2020_INEI/INEI_LIMITE_DISTRITAL.shp')

In [ ]:
maps

In [ ]:
#Check unique values
maps['CODIGO'].is_unique

In [ ]:
maps['CODIGO'].unique().size

In [ ]:
df_dengue.info()


In [ ]:
# Filtrar Año 2021
dengue_2021 = df_dengue[df_dengue.Año == 2021]
dengue_2021

In [ ]:
#Agrupar por Ubigeo
dengue_2021["Casos"] = dengue_2021["Casos"].astype('Int64')

dengue_2021

In [ ]:
#Máximo de casos de dengue en un distrito por semana
dengue_2021["Casos"].max()


In [ ]:
#Prueba en agrupar por ubigeo

dengue_max = dengue_2021[dengue_2021.Ubigeo == "010201"]
dengue_max.groupby( [ 'Año' ], as_index = False )[ [ 'Casos' ] ].sum()

In [ ]:
#AGRUPAR POR UBIGEO (distrito)

dengue_2021_1 = dengue_2021.groupby( [ 'Ubigeo' ], as_index = False )[ [ 'Casos' ] ].sum()
dengue_2021_1

In [ ]:
# Máximo de casos por distrito
dengue_2021_1["Casos"].max()

In [ ]:
#verificar dtypes de ambas bases antes de hacer merge
print(dengue_2021_1.dtypes)
print("")
print(maps.dtypes)

In [ ]:
# Convertir ambos identificadores del merge a mismo tipo
dengue_2021_1['Ubigeo'] = dengue_2021_1['Ubigeo'].astype('int64')
maps['UBIGEO'] = maps['UBIGEO'].astype('int64')

print(dengue_2021_1.dtypes)
print(maps.dtypes)

In [ ]:
#Merge datasets
dg_map1 = maps.merge(dengue_2021_1, left_on='UBIGEO', right_on='Ubigeo', how='left')

#Check if it is correct
dg_map1["Casos"].max()

In [ ]:
# Verify dtypes
dg_map1.dtypes

In [ ]:
# Change "Casos" to a readable type by geopandas
dg_map1["Casos"] = dg_map1["Casos"].astype("float64")
dg_map1['Casos'].dtype

In [ ]:
# First always check the distribution|
fig, ax = plt.subplots(figsize=(10, 10))
dg_map1["Casos"].hist(bins = 100)

#check the distribution of the variables BEFORE PLOTTING

In [ ]:
cmap = plt.cm.OrRd
dg_map1.plot( figsize = (20 , 20 ), 
             column = 'Casos', 
             cmap = cmap, 
             linestyle = 'dotted', 
             edgecolor = 'black', 
             legend = True, 
             missing_kwds = dict( color = '#D0D0D0' ) )
plt.show()

4. Use geopandas to plot the number of cases in 2021 by the province using a continuous legend. Do not forget to indicate the color of NA values. Use the provided shapefile. For this task, you will have to aggregate shapefiles at the province level.


In [ ]:
# Select the relevant columns from the latter dataset
dg_map2 = dg_map1[['IDPROV', "geometry", "Casos"]]
dg_map2

In [ ]:
# Use "dissolve" method to aggregate by "IDPROV"
prov = dg_map2.dissolve(by='IDPROV', aggfunc='sum')

#Replace 0 with NaN
prov.replace(0, np.nan, inplace=True)
prov

In [ ]:
#Plot
cmap = plt.cm.OrRd
prov.plot( figsize = (20 , 20 ), 
             column = 'Casos', 
             cmap = cmap, 
             linestyle = 'dotted', 
             edgecolor = 'black', 
             legend = True, 
             missing_kwds = dict( color = '#D0D0D0' ) )
plt.show()

5. Use geopandas to plot the number of cases by the department for all the years using subplots. Every subplot for each year. Do not forget to indicate the color of NA values. Use the provided shapefile. For this task, you will have to aggregate shapefiles at the department level.


In [ ]:
maps_6 = maps[['CCDD', 'geometry']]
maps_6 = maps_6.dissolve( by = 'CCDD' ).reset_index()
maps_6 = maps_6.rename(columns={'CCDD':'UBIDPTO'})
maps_6

In [ ]:
dpt_dengue6 = maps_6.merge(df_dengue)
dpt_dengue6

In [ ]:
dpt_dengue_years = dpt_dengue6.iloc[:,[0,2,9]]
dpt_dengue_years

In [ ]:
# Convert the column 'data' to numeric, forcing non-convertible to NaN
dpt_dengue_years['Casos'] = pd.to_numeric(dpt_dengue_years['Casos'], errors='coerce')

In [ ]:
dpt_dengue_years1 = dpt_dengue_years.groupby(['UBIDPTO','Año']).sum().reset_index()
dpt_dengue_years1

In [ ]:
dpt_dengue_years2 = dpt_dengue_years1.merge(maps_6)
dpt_dengue_years2

In [ ]:
# Si 'dist_2021_2' es un DataFrame y 'geometry' es el nombre de tu columna de geometría
dpt_dengue_years2 = gpd.GeoDataFrame(dpt_dengue_years2, geometry='geometry')

In [ ]:
fig, axis = plt.subplots( nrows = 3, ncols= 3, figsize = ( 10, 10 ) )

idx = 0
for i in range( 3 ):
    for j in range ( 3 ):
        
        
        ax = axis[ i ][ j ]
        
        Año = dpt_dengue_years2.Año.unique()[ idx ]
        
        df_den = dpt_dengue_years2[ dpt_dengue_years2.Año == Año ]
        
        df_den.plot( column='Casos', 
                  cmap='Reds', 
                  linestyle='--',
                  edgecolor='black', 
                  legend = True, 
                  missing_kwds= dict(color = "#DADADB"),
                  ax = ax 
                )
        
        ax.set_title( Año )
        
        idx = idx + 1

6. Use geopandas to plot the number of cases by the department for all 2021 quarters using subplots. Every subplot for each quarter. Use a categorical legend with 5 bins. Do not forget to indicate the color of NA values. Use the provided shapefile. For this task, you will have to aggregate shapefiles at the department level. Hint: Use Semana variable to group by quarters.


In [ ]:
# Filter year 2021
#6. Use geopandas to plot the number of cases by the department for all 2021 quarters using subplots. Every subplot for each quarter. Use a categorical legend with 5 bins. Do not forget to indicate the color of NA values. Use the provided shapefile. For this task, you will have to aggregate shapefiles at the department level. Hint: Use Semana variable to group by quarters.
dengue_2021_qt = dpt_dengue6[dpt_dengue6.Año == 2021]
dengue_2021_qt

In [ ]:
dengue_2021_qt2 = dengue_2021_qt.iloc[:,[0,3,9]]
dengue_2021_qt2

In [ ]:
dengue_2021_qt2 = dengue_2021_qt.iloc[:,[0,3,9]]
dengue_2021_qt2

In [ ]:
def assign_quarter(row):     # Calculates the quarter based on the value of 'Week'.
    quarter = ((row['Semana'] + 12) // 13) # 12 is added to the numerator to correctly handle weeks from 1 to 52.
    quarter = min(quarter, 4)
    return f'Q{quarter}'

In [ ]:
dengue_2021_qt2['Trim'] = dengue_2021_qt2.apply(lambda row: assign_quarter(row), axis=1)

In [ ]:
dengue_2021_qt2

In [ ]:
# Convert the column 'data' to numeric, forcing non-convertible to NaN
dengue_2021_qt2['Casos'] = pd.to_numeric(dengue_2021_qt2['Casos'], errors='coerce')

In [ ]:
dengue_2021_qt3 = dengue_2021_qt2.groupby(['UBIDPTO','Trim']).sum().reset_index()
dengue_2021_qt3

In [ ]:
dengue_2021_qt4 = pd.merge(dengue_2021_qt3, maps_6, on = ['UBIDPTO'], how = 'outer')
dengue_2021_qt5 = dengue_2021_qt4.sort_values('UBIDPTO').reset_index(drop=True)
dengue_2021_qt6 = dengue_2021_qt5.drop(columns = "Semana")
dengue_2021_qt6

In [ ]:
fig, axis = plt.subplots( nrows = 2, ncols= 2, figsize = ( 10, 10 ) )

idx = 0
for i in range( 2 ):
    for j in range ( 2 ):
        
        
        ax = axis[ i ][ j ]
        
        Qt = dengue_2021_qt6.Trim.unique()[ idx ]
        
        df_den1 = dengue_2021_qt6[ dengue_2021_qt6.Trim == Qt ]
        
        df_den.plot( column='Casos', 
                  cmap='Reds', 
                  linestyle='--',
                  edgecolor='black', 
                  legend = True,
                  missing_kwds= dict(color = "#DADADB"),
                  ax = ax 
                )
        
        ax.set_title( Qt )
        
        idx = idx + 1